In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
#import the code
from ML_LC_Classifier import mosaic_rasters
#define the unmosaicked raster paths
prob_2025_path = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\raw_prob_2025\*.tif"
prob_2020_path = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\raw_prob_2020\*.tif"
prob_2015_path = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\raw_prob_2015\*.tif"
prob_2005_path = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\raw_prob_2025\*.tif"

In [2]:
from pathlib import Path
from ML_LC_Classifier import heal_probability_seams
#define the paths for the probability mosaics
prob_mosaic_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\mosaic_probability\Landsat_prob_mosaic_2020.tif")
prob_mosaic_2015 = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\mosaic_probability\Landsat_prob_mosaic_2015.tif")
prob_mosaic_2005 = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\mosaic_probability\Landsat_prob_mosaic_2005.tif")
#define the paths for the healed mosaics (fixed seams)
healed_mosaic_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2020_Landsat.tif")
healed_mosaic_2015 = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2015_Landsat.tif")
healed_mosaic_2005 = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2005_Landsat.tif")
#run the function to heal the seams in the probability mosaics
fix_2020 = heal_probability_seams(input_path=prob_mosaic_2020, output_path=healed_mosaic_2020)
fix_2015 = heal_probability_seams(input_path=prob_mosaic_2015, output_path=healed_mosaic_2015)
fix_2005 = heal_probability_seams(input_path=prob_mosaic_2005, output_path=healed_mosaic_2005)
print(f"Healed mosaics saved to: {healed_mosaic_2020}, {healed_mosaic_2015}, {healed_mosaic_2005}")

Healed mosaics saved to: C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2020_Landsat.tif, C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2015_Landsat.tif, C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2005_Landsat.tif


In [3]:
from ML_LC_Classifier.hybrid_assembly import process_tile
#Setup the input and output
#define all the paths (fixed mosaics probability)
mosaic_prob_2025 = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2025_Landsat.tif"
mosaic_prob_2020 = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2020_Landsat.tif"
mosaic_prob_2015 = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2015_Landsat.tif"
mosaic_prob_2005 = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2005_Landsat.tif"
#segments and output
segments = r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Segments\TL_SNIC_segments_Landsat_30m.tif"
output_dir = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed")
#Define custom class weights (Scale: 0.1 to 5.0)
#Higher = trust segment more (smoother); Lower = trust pixel more (sharper)
landsat_smoothing_param = {
    'Primary_forest':        4.5,
    'Secondary_forest':      3.5,
    'Coastal_forest':        3.5,
    'Mangrove_forest':       2.5,
    'Teak_plantation':       1.5,
    'Eucalyptus_plantation': 1.5,
    'Coffe_agroforestry':    1.8,
    'Cocoa_agroforestry':    1.5,
    'Coconut_agroforestry':  1.5,
    'Mixed_garden':          1.2,
    'Paddy_field':           1.5,
    'Other_cropland':        1.0,
    'Grassland':             1.5,
    'Shrubland':             1.5,
    'Cleared_land':          1.0,
    'Built-up':              0.7,
    'Waterbody':             4.0,
    'Dry_riverbed':          0.5}

In [ ]:
#run the entire processing chain
print("Stage 1 processing 2025")
stage1_lc_2025 = process_tile(mosaic_prob_2025, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.65,
            smoothness_dict=landsat_smoothing_param)
print("Stage 1 processing 2020")
stage1_lc_2020 = process_tile(mosaic_prob_2020, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.65,
            smoothness_dict=landsat_smoothing_param)
print("Stage 1 processing 2015")
stage1_lc_2015 = process_tile(mosaic_prob_2015, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.65,
            smoothness_dict=landsat_smoothing_param)
print("Stage 1 processing 2005")
stage1_lc_2005 = process_tile(mosaic_prob_2005, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.65,
            smoothness_dict=landsat_smoothing_param)

Stage 1 processing 2025

--- Processing: Fix_mosaic_2025_Landsat ---
Probabilities: C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Fix_mosaic\Fix_mosaic_2025_Landsat.tif
Segments:      C:\AFLICM_Test\hybrid_lc\Landsat_Multi\Segments\TL_SNIC_segments_Landsat_30m.tif
Block size:    2048x2048 (overlap: 64px)
Stability tau: 0.65 | Adaptive sensitivity: 0.65
Active Mask-Layer Rules: 2
  1. False Built-up (16) in River Corridor -> Water (17)
  2. False Water ((17, 18)) in Confirmed Settlement -> Built-up (16)
Total windowed blocks: 21 (3 rows x 7 cols)
  Block  10/21 done ( 47.6%)...

--- Processing Complete! ---
Outputs written to: C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed
  1. LULC Map:        C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\Fix_mosaic_2025_Landsat_LULC.tif
  2. Stability Map:   C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\Fix_mosaic_2025_Landsat_Stability_Map.tif
  3. Confusion Index: C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output

In [6]:
from pathlib import Path
from ML_LC_Classifier.lc_post_process import apply_mmu
#Calibrated for Landsat 30m (1 px = 0.09 ha)
mmu_class_landsat = {
    1: 10,  # Primary_forest:        5 px (~0.45 ha, aligns with FAO 0.5 ha forest definition)
    2: 10,  # Secondary_forest:      5 px (~0.45 ha)
    3: 5,  # Mangrove_forest:       2 px (~0.18 ha, preserves narrow coastal/estuarine fringes)
    4: 5,  # Coastal_forest:        3 px (~0.27 ha)
    5: 3,  # Teak_plantation:       3 px (~0.27 ha)
    6: 2,  # Eucalyptus_plantation: 2 px (~0.18 ha)
    7: 10,  # Coffe_agroforestry:    3 px (~0.27 ha, preserves shaded mountain groves)
    8: 5,  # Cocoa_agroforestry:    3 px (~0.27 ha)
    9: 10,  # Coconut_agroforestry:  2 px (~0.18 ha, coastal strips)
    10: 5,  # Mixed_garden:          2 px (~0.18 ha, smallholder homestead gardens)
    11: 5,  # Paddy_field:           2 px (~0.18 ha, small valley terraced paddies)
    12: 5,  # Other_cropland:        2 px (~0.18 ha)
    13: 10,  # Grassland:             3 px (~0.27 ha)
    14: 10,  # Shrubland:             3 px (~0.27 ha)
    15: 10,  # Cleared_land:          3 px (~0.27 ha, slash-and-burn clearings)
    16: 10,  # Built-up:              2 px (~0.18 ha, preserves rural villages/hamlets)
    17: 10,  # Waterbody:             5 px (~0.45 ha, ponds/lakes)
}
input_folder = Path(r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov")
output_folder = input_folder / "MMU_processed_Landsat"
output_folder.mkdir(exist_ok=True)

raster_files = sorted(
    list(input_folder.glob("*.tif")) +
    list(input_folder.glob("*.tiff"))
)

outputs = list(map(
    lambda src: apply_mmu(
        src,
        output_folder / f"{src.stem}_Filtered.tif",
        table=mmu_class_landsat,
        iterations=5,
    ),
    raster_files,
))

outputs

[WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/Fix_mosaic_2005_Landsat_LULC_Filtered.tif'),
 WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/Fix_mosaic_2015_Landsat_LULC_Filtered.tif'),
 WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/Fix_mosaic_2020_Landsat_LULC_Filtered.tif'),
 WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/Fix_mosaic_2025_Landsat_LULC_Filtered.tif')]

In [8]:
from ML_LC_Classifier import analyze_multi_temporal_transitions, export_multi_epoch_sankey

# 1. Define your 4 epoch classified rasters
epoch_rasters = {
    2005: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2005_Landsat_LULC_Filtered.tif",
    2015: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2015_Landsat_LULC_Filtered.tif",
    2020: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2020_Landsat_LULC_Filtered.tif",
    2025: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2025_Landsat_LULC_Filtered.tif",
}

# 2. Run multi-temporal transition & trajectory analysis
result = analyze_multi_temporal_transitions(
    epoch_rasters=epoch_rasters,
    output_trajectory_raster=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\trajectory_map_2005_2025.tif",
    output_csv_dir=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\transition_tables",
    block_size=2048)

# 3. Print trajectory breakdown (Hectares and % of Timor-Leste)
print(result.trajectory_counts)

# 4. View 2015 -> 2020 transition summary (Gains, Losses, Annual Rates)
print(result.interval_summaries[1].class_summary_ha)

# 5. Generate interactive Sankey diagram for your thesis/paper
export_multi_epoch_sankey(
    result=result,
    output_html_path="output/landcover_sankey_2005_2025.html",
    min_area_ha=50.0,
)

                              Trajectory_Category     Area_ha  Share_pct
Trajectory_ID                                                           
1                      Persistent / Stable Matrix  2101306.86      84.42
2                      Unidirectional Forest Loss        0.00       0.00
3                    Forest Regrowth / Succession        0.00       0.00
4               Rotational Swidden / Fallow Cycle    17760.42       0.71
5              Agroforestry / Plantation Adoption        0.00       0.00
6                      Urban / Built-up Expansion        0.00       0.00
7                       Water / Riverbed Dynamics        0.00       0.00
8                 Other Multi-Directional Changes    33055.65       1.33
9              Potential Spurious Noise (QA Flag)   336860.64      13.53
                       Area_2015_ha  Area_2020_ha  Persistence_ha   Loss_ha  \
Class                                                                         
Primary_forest            182258.73    

WindowsPath('output/landcover_sankey_2005_2025.html')

In [1]:
from ML_LC_Classifier import (apply_multi_epoch_transition_consistency, analyze_multi_temporal_transitions, export_multi_epoch_sankey)
# 1. Define your 4 epoch classified rasters
epoch_rasters = {
    2005: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2005_Landsat_LULC_Filtered.tif",
    2015: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2015_Landsat_LULC_Filtered.tif",
    2020: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2020_Landsat_LULC_Filtered.tif",
    2025: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\Fix_mosaic_2025_Landsat_LULC_Filtered.tif",}
# STEP 1: Generate temporally consistent 2005-2025 maps (*_consistent.tif)
# anchor="last" uses 2025 as the reference truth and cascades backward:
# 2025 -> 2020 -> 2015 -> 2005
consistent_epoch_rasters = apply_multi_epoch_transition_consistency(
    epoch_rasters=epoch_rasters,
    output_dir=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistency",
    anchor="last",
)
print("Corrected maps saved at:", consistent_epoch_rasters)
# STEP 2: Run Trajectory & Pontius Analysis on the CORRECTED maps
result = analyze_multi_temporal_transitions(
    epoch_rasters=consistent_epoch_rasters,
    output_trajectory_raster=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\trajectory_map_2005_2025.tif",
    output_csv_dir=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\transition_tables",
)
# STEP 3: Export Interactive Multi-Epoch Sankey HTML
export_multi_epoch_sankey(
    result=result,
    output_html_path=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\sankey_diagram_2005_2025.html",
)

Corrected maps saved at: {2005: WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/consistency/Fix_mosaic_2005_Landsat_LULC_Filtered_consistent.tif'), 2015: WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/consistency/Fix_mosaic_2015_Landsat_LULC_Filtered_consistent.tif'), 2020: WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/consistency/Fix_mosaic_2020_Landsat_LULC_Filtered_consistent.tif'), 2025: WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/consistency/Fix_mosaic_2025_Landsat_LULC_Filtered_consistent.tif')}


WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/sankey_diagram_2005_2025.html')

In [2]:
from ML_LC_Classifier import analyze_multi_temporal_transitions, export_multi_epoch_sankey

#1. define the consistent epoch rasters (after applying temporal consistency)
epoch_rasters = {
    2025: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistency\LULC_2025_Landsat_consistent.tif",
    2020: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistency\LULC_2020_Landsat_consistent.tif",
    2015: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistency\LULC_2015_Landsat_consistent.tif",
    2005: r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistency\LULC_2005_Landsat_consistent.tif",
}

#2. Run multi-temporal transition & trajectory analysis
result = analyze_multi_temporal_transitions(
    epoch_rasters=epoch_rasters,
    output_trajectory_raster=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistent_trajectory_map_2005_2025.tif",
    output_csv_dir=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistent_transition_tables",
    block_size=2048)

# 3. Print trajectory breakdown (Hectares and % of Timor-Leste)
print(result.trajectory_counts)

# 4. View 2015 -> 2020 transition summary (Gains, Losses, Annual Rates)
print(result.interval_summaries[1].class_summary_ha)

# 5. Generate interactive Sankey diagram for your thesis/paper
export_multi_epoch_sankey(
    result=result,
    output_html_path=r"C:\AFLICM_Test\hybrid_lc\Landsat_Multi\stage1_output_healed\LandCov\MMU_processed_Landsat\consistent_transition_tables\landcover_sankey_2005_2025_consistent.html",
    min_area_ha=50.0,
)

                              Trajectory_Category     Area_ha  Share_pct
Trajectory_ID                                                           
1                      Persistent / Stable Matrix  2148034.41      86.30
2                      Unidirectional Forest Loss      753.03       0.03
3                    Forest Regrowth / Succession        0.00       0.00
4               Rotational Swidden / Fallow Cycle    17760.42       0.71
5              Agroforestry / Plantation Adoption     3527.19       0.14
6                      Urban / Built-up Expansion    10415.43       0.42
7                       Water / Riverbed Dynamics        0.00       0.00
8                 Other Multi-Directional Changes    29439.09       1.18
9              Potential Spurious Noise (QA Flag)   279054.00      11.21
                       Area_2015_ha  Area_2020_ha  Persistence_ha   Loss_ha  \
Class                                                                         
Primary_forest            225633.96    

WindowsPath('C:/AFLICM_Test/hybrid_lc/Landsat_Multi/stage1_output_healed/LandCov/MMU_processed_Landsat/consistent_transition_tables/landcover_sankey_2005_2025_consistent.html')